# M07 · Field-oriented control - Clarke, Park, current loops and SVM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M07_field_oriented_control_clarke_park_current_loops.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 70 min

**Skill Hub concepts:** Clarke and Park transforms; Field-oriented control; Space vector modulation and back-EMF

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M07")

## Map: the FOC pipeline running every PWM period (20 kHz)

```
 ia, ib, ic ──Clarke──> iα, iβ ──Park(θe)──> id, iq ──PI + decoupling──> vd, vq ──inverse Park──> vα, vβ ──SVM──> 3 duties
    (ADC)                                       ^ id_ref = 0 (no flux weakening), iq_ref = torque / (1.5 p λ)
                                    θe = p · θmech from the encoder ──────────────────────────────────────┘
```

Amplitude-invariant Clarke and Park:
$$i_\alpha = \tfrac{2}{3}\big(i_a - \tfrac12 i_b - \tfrac12 i_c\big), \quad i_\beta = \tfrac{1}{\sqrt3}(i_b - i_c)$$
$$i_d = i_\alpha\cos\theta + i_\beta\sin\theta, \qquad i_q = -i_\alpha\sin\theta + i_\beta\cos\theta$$

Surface PMSM in the rotor frame ($L_d = L_q = L$), with back-EMF $\omega_e\lambda$:
$$L\,\dot i_d = v_d - R i_d + \omega_e L i_q, \qquad L\,\dot i_q = v_q - R i_q - \omega_e L i_d - \omega_e \lambda, \qquad
\tau = \tfrac32\, p\, \lambda\, i_q$$

| Piece | Why |
|---|---|
| Rotor frame | sinusoidal phase currents become DC values, so plain PI controllers track them with zero error |
| Decoupling feed-forward | cancels the $\omega_e L$ cross terms and the back-EMF, so d and q behave like two independent RL circuits |
| PI gains from bandwidth $\omega_c$ | $k_p = L\,\omega_c$, $k_i = R\,\omega_c$ (pole-zero cancellation) |
| Space vector modulation | adding the common-mode offset $-(\max + \min)/2$ to the phase voltages reaches $V_{dc}/\sqrt3$ instead of $V_{dc}/2$ (15% more) |
| Back-EMF limit | top speed is where $\omega_e\lambda$ approaches the available $V_{dc}/\sqrt3$ |

In [ ]:
import math
import numpy as np

## Task 1: the transforms

In [ ]:
def clarke(ia, ib, ic):
    raise NotImplementedError  # TODO

def park(alpha, beta, theta):
    raise NotImplementedError  # TODO

def inv_park(d, q, theta):
    raise NotImplementedError  # TODO

lab.check("transforms", clarke, park, inv_park)

In [ ]:
%%writefile m07_transforms.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m07.h
// Grade it with lab.check("transforms", "m07_transforms.c"). For C++, name the file .cpp in both places.
#include "m07.h"

void clarke(double ia, double ib, double ic, double *alpha, double *beta) {
    (void)ia; (void)ib; (void)ic; *alpha = 0; *beta = 0; /* TODO */
}

void park(double alpha, double beta, double theta, double *d, double *q) {
    (void)alpha; (void)beta; (void)theta; *d = 0; *q = 0; /* TODO */
}

void inv_park(double d, double q, double theta, double *alpha, double *beta) {
    (void)d; (void)q; (void)theta; *alpha = 0; *beta = 0; /* TODO */
}

In [ ]:
# lab.check("transforms", "m07_transforms.c")  # uncomment after writing the C/C++ cell above

## Task 2: space vector modulation (min-max injection)
From $(v_\alpha, v_\beta)$: inverse Clarke to phase voltages, add $-(\max+\min)/2$, then duty $= 0.5 + v/V_{dc}$,
clamped to [0, 1].

In [ ]:
def svm(v_alpha, v_beta, vdc):
    raise NotImplementedError  # TODO

lab.check("svm", svm)

In [ ]:
%%writefile m07_svm.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m07.h
// Grade it with lab.check("svm", "m07_svm.c"). For C++, name the file .cpp in both places.
#include "m07.h"

void svm(double v_alpha, double v_beta, double vdc, double *duty) {
    (void)v_alpha; (void)v_beta; (void)vdc;
    duty[0] = duty[1] = duty[2] = 0.5; /* TODO: inverse Clarke, min-max offset, scale by vdc, clamp */
}

In [ ]:
# lab.check("svm", "m07_svm.c")  # uncomment after writing the C/C++ cell above

## Task 3: the current controller
PI on $i_d$ and $i_q$ plus decoupling: $v_d = \mathrm{PI}_d - \omega_e L i_q$, $v_q = \mathrm{PI}_q + \omega_e(L i_d + \lambda)$.
Limit the voltage vector magnitude to `v_max` (scale both, and do not integrate while limited).

In [ ]:
class FocCurrentController:
    def __init__(self, kp, ki, L, lam, dt, v_max):
        self.kp, self.ki, self.L, self.lam, self.dt, self.v_max = kp, ki, L, lam, dt, v_max
        self.id_int = self.iq_int = 0.0

    def update(self, id_ref, iq_ref, i_d, i_q, w_e):
        raise NotImplementedError  # TODO

lab.check("current_loop", FocCurrentController)

In [ ]:
%%writefile m07_current_loop.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/m07.h
// Grade it with lab.check("current_loop", "m07_current_loop.c"). For C++, name the file .cpp in both places.
#include "m07.h"

void foc_init(FocCurrentController *c, double kp, double ki, double L, double lam, double dt, double v_max) {
    c->kp = kp; c->ki = ki; c->L = L; c->lam = lam; c->dt = dt; c->v_max = v_max; c->id_int = c->iq_int = 0;
}

void foc_update(FocCurrentController *c, double id_ref, double iq_ref, double i_d, double i_q, double w_e, double *vd, double *vq) {
    (void)c; (void)id_ref; (void)iq_ref; (void)i_d; (void)i_q; (void)w_e;
    *vd = 0; *vq = 0; /* TODO: PI on d and q, decoupling, back-EMF, limit |v| without integrating when limited */
}

In [ ]:
# lab.check("current_loop", "m07_current_loop.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** $p = 4$ pole pairs, $\lambda = 0.01$ Wb, $i_q = 5$ A. Torque in N·m?

**M2 (Apply).** Same motor at 3000 rpm. Back-EMF amplitude $\omega_e \lambda$ in volts?

**M3 (Apply).** Largest phase-voltage amplitude SVM can produce linearly from a 24 V bus?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why run the current PI controllers in the rotor (dq) frame instead of on the phase currents?
- a) fewer ADC channels
- b) in the rotor frame the steady-state currents are constants, so PI controllers reach zero error at any speed; on phases they chase sine waves and lag more as speed rises
- c) it removes the need for an encoder

**K2 (Analyze).** Using M2 and M3, explain why this motor tops out near 3300 rpm on 24 V, and the two ways to go faster.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m07.h`).
3. In `roboprep/checks/m07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()